# BRISC Tumor-Type Classifier

Classifies glioma / meningioma / pituitary / non_tumor. The segmentation model (Attention Res-U-Net, see `03_camus_attnunet.ipynb`-style training) only predicts tumor-vs-background; this is a separate, small from-scratch CNN trained on the tumor type already encoded in BRISC's filenames (see `iteris.ingestion.build_brisc_dicts`).

## Kaggle setup
1. Add datasets: `briscdataset/brisc2025` + `iteris-pkg`
2. Accelerator: GPU T4 x2 (or CPU — this model is tiny, CPU works fine too, just slower)
3. Persistence: Files only
4. Add Secret `HF_TOKEN` (write-scope token) if you want the last cell to auto-push to HF Hub
5. Save Version → Save & Run All (Commit)

## 0 · Install (no kernel restart needed)

In [ ]:
import subprocess
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
REQ      = PKG_ROOT / 'requirements.txt'

subprocess.run(['pip', 'install', '-r', str(REQ), '--quiet'], check=True)
subprocess.run(['pip', 'install', 'huggingface_hub', '--quiet'], check=True)
print('Installed.')

## 1 · Load config + locate the BRISC dataset

In [ ]:
import sys
from pathlib import Path

init_files = list(Path('/kaggle/input').rglob('iteris/__init__.py'))
if not init_files:
    raise RuntimeError('iteris-pkg not attached.')
PKG_ROOT = init_files[0].parent.parent
sys.path.insert(0, str(PKG_ROOT))

from iteris.config import load_config
from iteris.utils  import get_device

cfg = load_config(str(PKG_ROOT / 'configs' / 'BRISC' / 'brisc_classifier.yaml'))

brisc_dirs = [p for p in Path('/kaggle/input').rglob('*') if p.is_dir() and p.name.lower() == 'brisc2025']
if brisc_dirs:
    cfg['data_root'] = str(brisc_dirs[0])
cfg['checkpoint_dir'] = '/kaggle/working'

print(f'Package root : {PKG_ROOT}')
print(f'Data root    : {cfg["data_root"]}')
print(f'Classes      : {cfg["num_classes"]} — {cfg["class_names"]}')
print(f'Epochs       : {cfg["epochs"]}  batch {cfg["batch_size"]}  lr {cfg["lr"]}')
print(f'Device       : {get_device()}')

## 2 · Train

In [ ]:
from iteris.classifier import run_classifier_training

result       = run_classifier_training(cfg)
model        = result['model']
history      = result['history']
best_val_acc = result['best_val_acc']
test_loader  = result['test_loader']
checkpoint   = result['checkpoint']

print(f'\n✓ Training done. Best val accuracy = {best_val_acc:.4f}  |  Checkpoint: {checkpoint}')

## 3 · Learning curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(history['train_loss'], label='train')
axes[0].plot(history['val_loss'], label='val')
axes[0].set_title('Loss'); axes[0].set_xlabel('epoch'); axes[0].legend()
axes[1].plot(history['val_acc'], color='green')
axes[1].set_title('Val accuracy'); axes[1].set_xlabel('epoch')
axes[1].axhline(0.85, color='gray', linestyle='--', linewidth=0.8)
plt.tight_layout()
plt.savefig('/kaggle/working/brisc_tumor_classifier_learning_curves.png', dpi=120)
plt.show()

## 4 · Test-set evaluation

In [ ]:
from iteris.classifier import evaluate_classifier_test_set

test_metrics = evaluate_classifier_test_set(model, test_loader, cfg)
print(f"Test accuracy: {test_metrics['accuracy']:.4f}  (n={test_metrics['n_test']})")
for cls, m in test_metrics['per_class'].items():
    print(f"  {cls:12s} precision={m['precision']:.3f}  recall={m['recall']:.3f}  f1={m['f1']:.3f}")

## 5 · Summary JSON

In [ ]:
from iteris.classifier import save_classifier_summary_json

summary_path = save_classifier_summary_json(history, test_metrics, cfg, best_val_acc)
print(f'Summary: {summary_path}')

## 6 · Push checkpoint to Hugging Face Hub

Requires a Kaggle Secret named `HF_TOKEN` (write-scope) — same one used for the segmentation checkpoints. Skips silently if the secret isn't attached.

In [ ]:
from huggingface_hub import HfApi, create_repo

try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('HF_TOKEN')
except Exception as exc:
    token = None
    print(f'No HF_TOKEN secret attached, skipping push ({exc}). Upload manually instead.')

if token:
    repo_id = 'Anfaal26/iteris-brisc-tumor-classifier'
    create_repo(repo_id, token=token, exist_ok=True)
    api = HfApi()
    for fname in ['brisc_tumor_classifier_best.pt', 'brisc_tumor_classifier_summary.json',
                  'brisc_tumor_classifier_learning_curves.png']:
        path = f'/kaggle/working/{fname}'
        if not Path(path).exists():
            print(f'MISSING: {path}')
            continue
        api.upload_file(path_or_fileobj=path, path_in_repo=fname, repo_id=repo_id, token=token)
        print(f'uploaded {fname} -> {repo_id}')